# Experiment: SepAware validation on Vigitel smoking and kidney mortality


Reproducible entry point for dataset 3 (Vigitel current smoking) and dataset 4 (365-day kidney-cohort mortality). The implementation is versioned in `JMIR_SepAware/experiments/run_strengthened_experiments.py`. Dialysis is deliberately excluded as an outcome.

Success requires held-out-real AUPRC improvement over the matched standard generator without ignoring minority coverage, diversity, calibration, or privacy diagnostics.


In [1]:
# Setup: imports and reproducibility
from __future__ import annotations

from pathlib import Path
import subprocess
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'JMIR_SepAware').exists():
    ROOT = ROOT.parents[1]
OUTPUT = ROOT / 'JMIR_SepAware' / 'experiments' / 'outputs'
SEED = 42
{'root': str(ROOT), 'output': str(OUTPUT), 'seed': SEED}


{'root': '/Users/User/Downloads/Qualification Experiment',
 'output': '/Users/User/Downloads/Qualification Experiment/JMIR_SepAware/experiments/outputs',
 'seed': 42}

## Plan

- Three repeated five-fold stratified partitions.
- Real only, class weighting, random over/undersampling, SMOTE, CTGAN, SepAware CTGAN, TVAE, and SepAware TVAE.
- Logistic regression, random forest, and CatBoost.
- Primary metric: held-out-real AUPRC. Secondary metrics: Macro-F1, minority metrics, AUROC, balanced accuracy, Brier score, and calibration.
- Diagnostics: minority coverage, diversity, exact matches, and nearest-synthetic membership proxy.


In [2]:
# Set RUN=True only to repeat the completed manuscript run. Checkpoints make restarts safe.
RUN = False
command = [str(ROOT / '.venv-jmir' / 'bin' / 'python'),
           str(ROOT / 'JMIR_SepAware' / 'experiments' / 'run_strengthened_experiments.py'),
           '--mode', 'full', '--datasets', 'vigitel_smoking', 'kidney_mortality',
           '--epochs', '30', '--append-existing', '--resume']
if RUN:
    subprocess.run(command, cwd=ROOT, check=True)
command


['/Users/User/Downloads/Qualification Experiment/.venv-jmir/bin/python',
 '/Users/User/Downloads/Qualification Experiment/JMIR_SepAware/experiments/run_strengthened_experiments.py',
 '--mode',
 'full',
 '--datasets',
 'vigitel_smoking',
 'kidney_mortality',
 '--epochs',
 '30',
 '--append-existing',
 '--resume']

## Results

The cells below read the authoritative completed exports. Results should be interpreted by task and against the corresponding standard generator. Cross-validation observations are paired and dependent.


In [3]:
summary = pd.read_csv(OUTPUT / 'predictive_metrics_summary.csv')
new = summary[summary['dataset'].isin(['Vigitel smoking', 'Kidney mortality'])]
new.groupby(['dataset', 'condition'])[['auprc_mean', 'macro_f1_mean', 'minority_recall_mean', 'brier_mean']].mean().round(4)


auprc_mean  macro_f1_mean  \
dataset          condition                                         
Kidney mortality Class weighting           0.1078         0.5253   
                 Random oversampling       0.1072         0.5285   
                 Random undersampling      0.1177         0.4895   
                 Real only                 0.1206         0.4862   
                 SMOTE                     0.0948         0.5207   
                 SepAware CTGAN            0.1117         0.5263   
                 SepAware TVAE             0.0975         0.4989   
                 Standard CTGAN            0.1114         0.5281   
                 Standard TVAE             0.0981         0.4983   
Vigitel smoking  Class weighting           0.2947         0.5741   
                 Random oversampling       0.2899         0.5816   
                 Random undersampling      0.2830         0.5583   
                 Real only                 0.2991         0.4969   
                 SMOTE                     0.2324         0.5397   
                 SepAware CTGAN            0.2578         0.5767   
                 SepAware TVAE             0.2588         0.5285   
                 Standard CTGAN            0.2206         0.5387   
                 Standard TVAE             0.2516         0.5247   

                                       minority_recall_mean  brier_mean  
dataset          condition                                               
Kidney mortality Class weighting                     0.3787      0.1565  
                 Random oversampling                 0.3692      0.1557  
                 Random undersampling                0.6202      0.2246  
                 Real only                           0.0000      0.0498  
                 SMOTE                               0.2555      0.1317  
                 SepAware CTGAN                      0.1707      0.0970  
                 SepAware TVAE                       0.0300      0.0616  
                 Standard CTGAN                      0.1558      0.0939  
                 Standard TVAE                       0.0254      0.0600  
Vigitel smoking  Class weighting                     0.4522      0.1645  
                 Random oversampling                 0.4658      0.1651  
                 Random undersampling                0.6517      0.2101  
                 Real only                           0.0327      0.0986  
                 SMOTE                               0.1894      0.1339  
                 SepAware CTGAN                      0.3056      0.1400  
                 SepAware TVAE                       0.1016      0.1100  
                 Standard CTGAN                      0.2339      0.1476  
                 Standard TVAE                       0.0978      0.1111

## Paired generator contrasts and diagnostics


In [4]:
long = pd.read_csv(OUTPUT / 'predictive_metrics_long.csv')
diag = pd.read_csv(OUTPUT / 'synthetic_diagnostics_long.csv')
rows = []
for dataset in ['Vigitel smoking', 'Kidney mortality']:
    for generator in ['CTGAN', 'TVAE']:
        keys = ['dataset', 'repeat', 'fold', 'classifier']
        selected = long[(long.dataset == dataset) & (long.condition == f'SepAware {generator}')]
        standard = long[(long.dataset == dataset) & (long.condition == f'Standard {generator}')]
        paired = selected.merge(standard, on=keys, suffixes=('_sep', '_std'))
        rows.append({'dataset': dataset, 'generator': generator, 'n': len(paired),
                     'AUPRC difference': (paired.auprc_sep - paired.auprc_std).mean(),
                     'Macro-F1 difference': (paired.macro_f1_sep - paired.macro_f1_std).mean()})
display(pd.DataFrame(rows).round(4))
display(diag[diag.dataset.isin(['Vigitel smoking', 'Kidney mortality'])].groupby(['dataset', 'condition'])[['minority_coverage_distance', 'minority_diversity', 'exact_match_rate', 'membership_auc']].mean().round(4))


,dataset,generator,n,AUPRC difference,Macro-F1 difference
0,Vigitel smoking,CTGAN,45,0.0372,0.0379
1,Vigitel smoking,TVAE,45,0.0073,0.0038
2,Kidney mortality,CTGAN,45,0.0003,-0.0017
3,Kidney mortality,TVAE,45,-0.0006,0.0006


minority_coverage_distance  \
dataset          condition                                    
Kidney mortality SepAware CTGAN                      1.6471   
                 SepAware TVAE                       3.1347   
                 Standard CTGAN                      1.7388   
                 Standard TVAE                       3.1682   
Vigitel smoking  SepAware CTGAN                      3.6195   
                 SepAware TVAE                       5.4433   
                 Standard CTGAN                      3.8073   
                 Standard TVAE                       5.2070   

                                 minority_diversity  exact_match_rate  \
dataset          condition                                              
Kidney mortality SepAware CTGAN              4.0895            0.0001   
                 SepAware TVAE               1.4364            0.0003   
                 Standard CTGAN              6.9078            0.0000   
                 Standard TVAE               2.0376            0.0001   
Vigitel smoking  SepAware CTGAN              6.1325            0.0000   
                 SepAware TVAE               1.2999            0.0000   
                 Standard CTGAN              7.3574            0.0000   
                 Standard TVAE               2.0887            0.0000   

                                 membership_auc  
dataset          condition                       
Kidney mortality SepAware CTGAN          0.5000  
                 SepAware TVAE           0.4986  
                 Standard CTGAN          0.5005  
                 Standard TVAE           0.5001  
Vigitel smoking  SepAware CTGAN          0.5089  
                 SepAware TVAE           0.5003  
                 Standard CTGAN          0.5009  
                 Standard TVAE           0.5005